In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../common/environmental_config

In [0]:
%run ../common/gold-helper

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
df_constructors = (spark.table(f"{catalog_name}.{silver_schema}.constructors").filter(F.col('batch_id') == v_batch_id))
df_ref_nationality_region = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
df_dim_constructors = (
    df_constructors.join(df_ref_nationality_region, df_constructors.nationality == df_ref_nationality_region.nationality, "left")
    .select(
        df_constructors.constructor_id,
        df_constructors.constructor_name,
        df_constructors.nationality,
        df_ref_nationality_region.region.alias("nationality_region")
        )
)

In [0]:
write_to_gold(
    input_df=df_dim_constructors,
    target_table=target_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "nationality_region"
    ]
)